# ⚡ Experiment 13 — Zener Diode: Voltage Regulation Study
**Course:** Electronics Lab | B.Sc. Physics (Third Year)  
**Institution:** Tri-Chandra Multiple Campus, Tribhuvan University  
**Author:** Nabin Pandey  
**Date:** November 2025

---

### Objectives
1. Study **line regulation** — how V_out behaves as V_in changes.
2. Study **load regulation** — how V_out behaves as load resistance changes.
3. Determine the **Zener breakdown voltage V_Z**.
4. Perform **error analysis** using uncertainty propagation.

---
> 📂 **Data sources:**  
> `data/zener_diode.csv` — voltage sweep  
> `data/zener_diode_resistance.csv` — load resistance sweep


## 1. Imports & Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

# ── Plot styling ──
plt.rcParams.update({
    'figure.dpi'     : 120,
    'axes.titlesize' : 13,
    'axes.labelsize' : 11,
    'axes.grid'      : True,
    'grid.alpha'     : 0.4,
    'lines.linewidth': 1.8,
    'font.family'    : 'DejaVu Sans',
})

# ── File paths (relative to repo root) ──
PATH_VOLTAGE    = 'data/zener_diode.csv'
PATH_RESISTANCE = 'data/zener_diode_resistance.csv'

# ── Create plots folder if it doesn't exist ──
os.makedirs('plots', exist_ok=True)

# ── Instrument uncertainty (DMM least count = 0.1 V, 0.1 ohm) ──
DELTA_V = 0.05   # +/- V
DELTA_R = 0.05   # +/- ohm

print('Setup complete.')
print(f'Voltage data    -> {PATH_VOLTAGE}')
print(f'Resistance data -> {PATH_RESISTANCE}')

## 2. Load Data

Reading CSV files from the `data/` folder.  
If running **locally**, keep the `data/` folder next to this notebook.  
If running on **Google Colab**, upload the CSV files or mount Drive.


In [ ]:
# ── Load voltage sweep data ──
df_v = pd.read_csv(PATH_VOLTAGE)
df_v = df_v.sort_values('V_in').reset_index(drop=True)

# ── Load resistance sweep data ──
df_r = pd.read_csv(PATH_RESISTANCE)
df_r = df_r.sort_values('load_resistance').reset_index(drop=True)

print('── Voltage Sweep  (zener_diode.csv) ──')
print(df_v.to_string(index=False))
print()
print('── Load Resistance Sweep  (zener_diode_resistance.csv) ──')
print(df_r.to_string(index=False))

## 3. Descriptive Statistics

In [ ]:
print('── Voltage Sweep Stats ──')
print(df_v.describe().round(3))
print()
print('── Load Resistance Sweep Stats ──')
print(df_r.describe().round(3))

## 4. Line Regulation Analysis

**What is line regulation?**  
It measures how stable V_out is when V_in changes.

$$\\text{Line Regulation} = \\frac{\\Delta V_{out}}{\\Delta V_{in}} \\quad [\\text{V/V}]$$

A perfect Zener regulator gives **0 V/V** in the breakdown zone.

We split the data into:
- **Pre-breakdown** (V_in < 9 V): V_out rises linearly, no regulation
- **Regulation zone** (V_in >= 9 V): V_out clamped at V_Z


In [ ]:
# ── Split regions ──
df_pre = df_v[df_v['V_in'] < 9].copy()
df_reg = df_v[df_v['V_in'] >= 9].copy()

# ── Linear fits ──
slope_pre, intercept_pre, r_pre, _, se_pre = linregress(df_pre['V_in'], df_pre['V_out'])
slope_full, intercept_full, r_full, _, se_full = linregress(df_v['V_in'], df_v['V_out'])

# ── Zener breakdown voltage ──
V_Z     = df_reg['V_out'].mean()
V_Z_std = df_reg['V_out'].std()

# ── Line regulation in regulated zone ──
delta_vout_reg = df_reg['V_out'].max() - df_reg['V_out'].min()
delta_vin_reg  = df_reg['V_in'].max()  - df_reg['V_in'].min()
line_reg       = delta_vout_reg / delta_vin_reg
delta_line_reg = (2 * DELTA_V) / delta_vin_reg

print(f'Zener Breakdown Voltage  V_Z = {V_Z:.2f} +/- {DELTA_V} V')
print()
print(f'Pre-breakdown fit (V_in = 0-8 V)')
print(f'  Slope     = {slope_pre:.4f} +/- {se_pre:.4f} V/V')
print(f'  Intercept = {intercept_pre:.4f} V')
print(f'  R2        = {r_pre**2:.4f}')
print()
print(f'Full-range fit (V_in = 0-12 V)')
print(f'  Slope     = {slope_full:.4f} +/- {se_full:.4f} V/V')
print(f'  R2        = {r_full**2:.4f}')
print(f'  NOTE: full-range fit spans both regions -- not meaningful for regulation alone')
print()
print(f'Line Regulation (regulated zone V_in = 9-12 V)')
print(f'  Delta_Vout = {delta_vout_reg:.2f} V   Delta_Vin = {delta_vin_reg:.2f} V')
print(f'  Line Reg   = {line_reg:.4f} +/- {delta_line_reg:.4f} V/V')
print(f'  Line Reg   = {line_reg*100:.2f}%')

### 4.1 Plot — V_out vs V_in (Line Regulation)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

# Measured data
ax.scatter(df_v['V_in'], df_v['V_out'],
           color='steelblue', s=70, zorder=5,
           edgecolors='white', linewidths=0.8, label='Measured data')

# Pre-breakdown fit
x_pre = np.linspace(0, 8.5, 200)
ax.plot(x_pre, slope_pre * x_pre + intercept_pre,
        '--', color='tomato', alpha=0.85,
        label=f'Pre-breakdown fit:  y = {slope_pre:.3f}x + {intercept_pre:.3f}')

# Full-range fit (reference)
x_all = np.linspace(0, 12, 200)
ax.plot(x_all, slope_full * x_all + intercept_full,
        ':', color='orange', alpha=0.6, linewidth=1.4,
        label=f'Full-range fit:  y = {slope_full:.3f}x + {intercept_full:.3f}')

# Breakdown voltage line
ax.axhline(V_Z, color='seagreen', linewidth=1.4, linestyle='-.',
           label=f'V_Z = {V_Z:.1f} V (breakdown voltage)')

# Shade regulation zone
ax.axvspan(9, 12, alpha=0.07, color='seagreen', label='Regulation zone (V_in >= 9 V)')

# Arrow annotation
ax.annotate('Breakdown onset\n(V_in = 9 V)',
            xy=(9, 7.1), xytext=(6.0, 5.0),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.2),
            fontsize=9, color='dimgray')

ax.set_xlabel('Input Voltage,  V_in  (V)')
ax.set_ylabel('Output Voltage,  V_out  (V)')
ax.set_title('Voltage Regulation — V_out vs V_in\n(Line Regulation Characteristic)')
ax.set_xlim(-0.3, 12.8)
ax.set_ylim(-0.3, 8.8)
ax.legend(fontsize=8.5, loc='upper left')
plt.tight_layout()
plt.savefig('plots/vout_vs_vin.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved -> plots/vout_vs_vin.png')

## 5. Load Regulation Analysis

**What is load regulation?**  
How much does V_out drop when we connect a heavier load (smaller R_L)?

$$\\text{Load Regulation (\\%)} = \\frac{V_{NL} - V_{FL}}{V_{FL}} \\times 100$$

- **V_NL** = no-load voltage (largest R_L)
- **V_FL** = full-load voltage (smallest practical R_L, nonzero output)

> R_L = 0 ohm (short circuit) is **excluded** — degenerate case, not a real load.


In [ ]:
# Exclude short-circuit point
df_r_valid = df_r[df_r['load_resistance'] > 0].copy()

idx_min = df_r_valid['load_resistance'].idxmin()
idx_max = df_r_valid['load_resistance'].idxmax()

R_FL = df_r_valid.loc[idx_min, 'load_resistance']
V_FL = df_r_valid.loc[idx_min, 'output_voltage']
R_NL = df_r_valid.loc[idx_max, 'load_resistance']
V_NL = df_r_valid.loc[idx_max, 'output_voltage']

# Load regulation
load_reg_pct = (V_NL - V_FL) / V_FL * 100

# Uncertainty propagation
delta_load_reg = load_reg_pct * np.sqrt(
    (DELTA_V / (V_NL - V_FL))**2 +
    (DELTA_V / (V_NL - V_FL))**2 +
    (DELTA_V / V_FL)**2
)

# Log-space linear fit
log_R = np.log10(df_r_valid['load_resistance'])
slope_log, intercept_log, r_log, _, se_log = linregress(log_R, df_r_valid['output_voltage'])

# Saturation zone (R_L >= 2.5 ohm)
df_sat = df_r_valid[df_r_valid['load_resistance'] >= 2.5]
V_sat_mean = df_sat['output_voltage'].mean()
V_sat_std  = df_sat['output_voltage'].std()

print(f'Full-load : R_L = {R_FL} ohm  ->  V_FL = {V_FL:.2f} V')
print(f'No-load   : R_L = {R_NL} ohm  ->  V_NL = {V_NL:.2f} V')
print()
print(f'Load Regulation = {load_reg_pct:.1f} +/- {delta_load_reg:.1f} %')
print()
print(f'Log-space fit   : V_out = {slope_log:.3f} * log10(R) + {intercept_log:.3f}')
print(f'                  R2 = {r_log**2:.4f}')
print(f'                  {slope_log:.3f} V per decade of R')
print()
print(f'Saturation zone (R_L >= 2.5 ohm): V_out = {V_sat_mean:.2f} +/- {V_sat_std:.2f} V')

### 5.1 Plot — V_out vs Load Resistance (Load Regulation)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

# Measured data
ax.scatter(df_r_valid['load_resistance'], df_r_valid['output_voltage'],
           color='steelblue', s=80, zorder=5,
           edgecolors='white', linewidths=0.8, label='Measured data')

# Short-circuit marker
ax.scatter([0.01], [0], color='gray', s=70, marker='x', linewidths=2.5,
           zorder=4, label='Short circuit R_L=0 (excluded)')

# Log-space fit curve
R_fit = np.logspace(np.log10(0.5), np.log10(4.5), 300)
V_fit = slope_log * np.log10(R_fit) + intercept_log
ax.plot(R_fit, V_fit, '--', color='tomato', alpha=0.85,
        label=f'Log fit:  V = {slope_log:.3f}*log(R) + {intercept_log:.3f}')

# V_Z reference line
ax.axhline(V_Z, color='seagreen', linewidth=1.4, linestyle='-.',
           label=f'V_Z = {V_Z:.1f} V')

# Shade good-regulation zone
ax.axvspan(2.5, 5.0, alpha=0.07, color='seagreen',
           label='Saturation zone (good regulation)')

# Annotations
ax.annotate(f'V_FL = {V_FL} V  (R_L = {R_FL} ohm)',
            xy=(R_FL, V_FL), xytext=(0.7, 2.0),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.2),
            fontsize=8.5, color='dimgray')
ax.annotate(f'V_NL = {V_NL} V  (R_L = {R_NL} ohm)',
            xy=(R_NL, V_NL), xytext=(2.8, 5.7),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.2),
            fontsize=8.5, color='dimgray')

ax.set_xscale('log')
ax.set_xlabel('Load Resistance,  R_L  (ohm)  [log scale]')
ax.set_ylabel('Output Voltage,  V_out  (V)')
ax.set_title('Voltage Regulation — V_out vs Load Resistance\n(Load Regulation Characteristic)')
ax.set_ylim(-0.5, 9.2)
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.savefig('plots/vout_vs_load_resistance.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved -> plots/vout_vs_load_resistance.png')

## 6. Combined Summary Figure

In [ ]:
fig = plt.figure(figsize=(14, 5.5))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

ax1 = fig.add_subplot(gs[0])
ax1.scatter(df_v['V_in'], df_v['V_out'],
            color='steelblue', s=60, zorder=5, edgecolors='white', linewidths=0.7)
ax1.plot(x_pre, slope_pre * x_pre + intercept_pre,
         '--', color='tomato', alpha=0.8, linewidth=1.5, label='Pre-breakdown fit')
ax1.axhline(V_Z, color='seagreen', linewidth=1.3, linestyle='-.', label=f'V_Z = {V_Z:.1f} V')
ax1.axvspan(9, 12, alpha=0.07, color='seagreen')
ax1.set_xlabel('V_in (V)')
ax1.set_ylabel('V_out (V)')
ax1.set_title('(a)  Line Regulation', fontweight='bold')
ax1.legend(fontsize=8)

ax2 = fig.add_subplot(gs[1])
ax2.scatter(df_r_valid['load_resistance'], df_r_valid['output_voltage'],
            color='steelblue', s=60, zorder=5, edgecolors='white', linewidths=0.7)
ax2.plot(R_fit, V_fit, '--', color='tomato', alpha=0.8, linewidth=1.5, label='Log fit')
ax2.axhline(V_Z, color='seagreen', linewidth=1.3, linestyle='-.', label=f'V_Z = {V_Z:.1f} V')
ax2.axvspan(2.5, 5.0, alpha=0.07, color='seagreen')
ax2.set_xscale('log')
ax2.set_xlabel('R_L (ohm)  [log scale]')
ax2.set_ylabel('V_out (V)')
ax2.set_title('(b)  Load Regulation', fontweight='bold')
ax2.legend(fontsize=8)

fig.suptitle('Experiment 13 — Zener Diode Voltage Regulation Summary',
             fontsize=13, fontweight='bold', y=1.02)
plt.savefig('plots/zener_summary.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved -> plots/zener_summary.png')

## 7. Error Analysis

### Instrument uncertainties
| Quantity | Instrument | Least Count | Uncertainty |
|----------|-----------|-------------|-------------|
| V_in, V_out | Digital Multimeter | 0.1 V | +/- 0.05 V |
| R_L | Resistance box | 0.1 ohm | +/- 0.05 ohm |

### Load regulation uncertainty formula

$$\\delta(\\text{LR}) = \\text{LR} \\times \\sqrt{ \\left(\\frac{\\delta V_{NL}}{V_{NL}-V_{FL}}\\right)^2 + \\left(\\frac{\\delta V_{FL}}{V_{NL}-V_{FL}}\\right)^2 + \\left(\\frac{\\delta V_{FL}}{V_{FL}}\\right)^2 }$$


In [ ]:
print('=' * 55)
print('       COMPLETE ERROR ANALYSIS SUMMARY')
print('=' * 55)
print()
print('1. Zener Breakdown Voltage')
print(f'   V_Z = {V_Z:.2f} +/- {DELTA_V} V')
print()
print('2. Line Regulation  (V_in = 9-12 V)')
print(f'   Delta_Vout = {delta_vout_reg:.2f} V   Delta_Vin = {delta_vin_reg:.2f} V')
print(f'   Line Reg   = {line_reg:.4f} +/- {delta_line_reg:.4f} V/V')
print(f'              = {line_reg*100:.2f} +/- {delta_line_reg*100:.2f} %')
print()
print(f'3. Load Regulation  (R_L: {R_FL} -> {R_NL} ohm)')
print(f'   V_FL = {V_FL:.2f} +/- {DELTA_V} V  (R_L = {R_FL} ohm)')
print(f'   V_NL = {V_NL:.2f} +/- {DELTA_V} V  (R_L = {R_NL} ohm)')
print(f'   Load Reg = {load_reg_pct:.1f} +/- {delta_load_reg:.1f} %')
print()
print('4. Fit Quality')
print(f'   Pre-breakdown  R2 = {r_pre**2:.4f}   slope = {slope_pre:.4f} +/- {se_pre:.4f} V/V')
print(f'   Log-space load R2 = {r_log**2:.4f}   slope = {slope_log:.4f} +/- {se_log:.4f} V/decade')
print()
print('5. Saturation Zone (R_L >= 2.5 ohm)')
print(f'   Mean V_out = {V_sat_mean:.3f} +/- {V_sat_std:.3f} V')
print(f'   Deviation from V_Z = {abs(V_sat_mean-V_Z):.3f} V  ({abs(V_sat_mean-V_Z)/V_Z*100:.1f}%)')
print()
print('=' * 55)

## 8. Final Results Table

In [ ]:
results = pd.DataFrame({
    'Quantity': [
        'Zener Breakdown Voltage (V_Z)',
        'Line Regulation (regulated zone)',
        'Load Regulation (full range)',
        'No-load Voltage V_NL',
        'Full-load Voltage V_FL',
        'Pre-breakdown slope (V/V)',
        'Log-fit slope (V/decade)',
    ],
    'Value': [
        f'{V_Z:.2f} V',
        f'{line_reg:.4f} V/V  =  {line_reg*100:.2f}%',
        f'{load_reg_pct:.1f}%',
        f'{V_NL:.2f} V',
        f'{V_FL:.2f} V',
        f'{slope_pre:.4f}',
        f'{slope_log:.4f}',
    ],
    'Uncertainty': [
        f'+/- {DELTA_V} V',
        f'+/- {delta_line_reg:.4f} V/V',
        f'+/- {delta_load_reg:.1f}%',
        f'+/- {DELTA_V} V',
        f'+/- {DELTA_V} V',
        f'+/- {se_pre:.4f} (SE)',
        f'+/- {se_log:.4f} (SE)',
    ]
})

print(results.to_string(index=False))

## 9. Conclusion

1. **V_Z = 7.1 +/- 0.05 V** confirmed from the flat regulated region (V_in = 9-12 V).

2. **Line regulation = 0.0 V/V** in the regulated zone — perfect clamping within instrument precision. Pre-breakdown slope ~0.63 V/V confirms the diode only regulates after entering breakdown.

3. **Load regulation = 111 +/- 2%** over full R_L range. Expected — at very low R_L, the load demands more current than the Zener can divert. For **R_L >= 2.5 ohm**, V_out stabilizes within 4% of V_Z.

4. **No-load overshoot** (V_out = 7.4 V vs V_Z = 7.1 V) is due to the Zener's finite dynamic impedance.

5. **Design rule:** Always keep R_L large enough so that Zener current stays above its minimum holding current.

---
*Nabin Pandey | Tri-Chandra Multiple Campus, Tribhuvan University*  
*Data: `data/` | Plots: `plots/` | Notebook: `zener_diode.ipynb`*
